# Activity 1: RAGAS Evaluation with Cost Analysis

This notebook builds two parallel RAG pipelines over the **cat-health-guide.pdf**:

1. **Fireworks AI** — `gpt-oss-20b` + `qwen3-embedding-8b`
2. **OpenAI** — `gpt-4.1-mini` + `text-embedding-3-small`

We evaluate both with RAGAS metrics and instrument with LangSmith for cost comparison.

### LangSmith Artifacts

| File | Description |
|---|---|
| [ragas_projects.png](data/ragas_projects.png) | LangSmith dashboard showing all four RAGAS tracing projects |
| [ragas_fireworks.png](data/ragas_fireworks.png) | LangSmith traces for the Fireworks RAG pipeline |
| [ragas_openai.png](data/ragas_openai.png) | LangSmith traces for the OpenAI RAG pipeline |
| [langsmith-ragas_fireworks_eval.json](data/langsmith-ragas_fireworks_eval.json) | Exported trace data for Fireworks RAGAS evaluation runs |
| [langsmith-ragas_openai_eval.json](data/langsmith-ragas_openai_eval.json) | Exported trace data for OpenAI RAGAS evaluation runs |

## 1. Setup

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Fireworks API Key: ")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

# Disable auto-tracing — we pass LangChainTracer callbacks explicitly
# with per-section project names instead
os.environ["LANGCHAIN_TRACING_V2"] = "false"

## 2. Load Data

In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

loader = DirectoryLoader("data/", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()
print(f"Loaded {len(docs)} pages from cat-health-guide.pdf")

Loaded 22 pages from cat-health-guide.pdf


## 3. Synthetic Test Data (RAGAS)

Generate a synthetic evaluation dataset from our documents using RAGAS.

In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_48930/4102110639.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_48930/4102110639.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [4]:
from langchain_core.documents import Document
from ragas.testset import TestsetGenerator

# PyMuPDFLoader creates one doc per page (22 pages). RAGAS HeadlineSplitter
# expects documents with clear heading structure, so we merge all pages into
# a single document to let RAGAS split on its own terms.
merged_doc = Document(
    page_content="\n\n".join(d.page_content for d in docs),
    metadata={"source": "data/cat-health-guide.pdf"},
)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs([merged_doc], testset_size=10)
dataset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/26 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,"In Texas cats, what kind of health stuff do we...",[VETERINARY PRACTICE GUIDELINES 2021 AAHA/AAFP...,The guidelines include a comprehensive table o...,Feline Healthcare Veterinarian,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,What is the significance of the AAFP guideline...,[Introduction The feline patient’s life stage ...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,Feline Healthcare Veterinarian,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer
2,What is the significance of JAAHA in the conte...,[Importance of Feline-Friendly Handling Both A...,The guidelines referenced are from the 2021 AA...,Feline Healthcare Veterinarian,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,What is AAHA?,"[requests such as, “What would you like to dis...",The provided context does not include specific...,Feline Healthcare Veterinarian,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
4,What does AAHA stand for?,[the examination by each life stage are listed...,The context does not specify what AAHA stands ...,Feline Healthcare Veterinarian,MISSPELLED,SHORT,single_hop_specific_query_synthesizer
5,What the American Association of Feline Practi...,[<1-hop>\n\nVETERINARY PRACTICE GUIDELINES 202...,The guidelines from the American Association o...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
6,How do environmental changes help cats with ur...,"[<1-hop>\n\ncat. Notably, there is evidence th...",Environmental modifications can be beneficial ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,How can I tell if my cat is eating and drinkin...,"[<1-hop>\n\n(current and past), clinical signs...",To assess if your cat is eating and drinking n...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,How does DJD relate to environmental managemen...,"[<1-hop>\n\ncat. Notably, there is evidence th...",The context indicates that environmental manag...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,How can environmental modifications and gentle...,"[<1-hop>\n\ncat. Notably, there is evidence th...","Environmental modifications, such as multimoda...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## 4. Shared RAG Configuration

Both pipelines use the same chunk size, retrieval k, and prompt template so the only variable is the model provider.

In [5]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from typing import TypedDict


def _tiktoken_len(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


CHUNK_SIZE = 750
CHUNK_OVERLAP = 0
RETRIEVAL_K = 4

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("human",
     "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
     "Use the provided context to answer the provided user query. "
     "Only use the provided context to answer the query. "
     'If you do not know the answer, or it\'s not contained in the provided context respond with "I don\'t know"')
])

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=_tiktoken_len
)
chunks = text_splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")


class RAGState(TypedDict):
    question: str
    context: list[Document]
    response: str

Split into 42 chunks


## 5. Fireworks AI RAG Pipeline

In [6]:
from langchain_openai import ChatOpenAI as _ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings as _OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"

fw_embeddings = _OpenAIEmbeddings(
    model=os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=4096,
)

fw_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks, embedding=fw_embeddings,
    location=":memory:", collection_name="fw_rag",
)
fw_retriever = fw_vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

fw_llm = _ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
)


def fw_retrieve(state: RAGState) -> RAGState:
    return {"context": fw_retriever.invoke(state["question"])}


def fw_generate(state: RAGState) -> RAGState:
    chain = RAG_PROMPT | fw_llm | StrOutputParser()
    response = chain.invoke({"query": state["question"], "context": state.get("context", [])})
    return {"response": response}


fw_graph = StateGraph(RAGState).add_sequence([fw_retrieve, fw_generate])
fw_graph.add_edge(START, "fw_retrieve")
fw_rag = fw_graph.compile()

# Quick smoke test
fw_test = fw_rag.invoke({"question": "What vaccinations do kittens need?"})
print(fw_test["response"][:200])

Kittens should be given the *core* feline vaccines and, depending on risk factors, the *FeLV* vaccine:

| Vaccine | Virus | Core status | Typical kitten‑series schedule |
|---|---|---|---|
| **Rabies*


## 6. OpenAI RAG Pipeline

In [7]:
oai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

oai_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks, embedding=oai_embeddings,
    location=":memory:", collection_name="oai_rag",
)
oai_retriever = oai_vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

oai_llm = ChatOpenAI(model="gpt-4.1-mini")


def oai_retrieve(state: RAGState) -> RAGState:
    return {"context": oai_retriever.invoke(state["question"])}


def oai_generate(state: RAGState) -> RAGState:
    chain = RAG_PROMPT | oai_llm | StrOutputParser()
    response = chain.invoke({"query": state["question"], "context": state.get("context", [])})
    return {"response": response}


oai_graph = StateGraph(RAGState).add_sequence([oai_retrieve, oai_generate])
oai_graph.add_edge(START, "oai_retrieve")
oai_rag = oai_graph.compile()

# Quick smoke test
oai_test = oai_rag.invoke({"question": "What vaccinations do kittens need?"})
print(oai_test["response"][:200])

Kittens need core vaccinations including rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and feline panleukopenia virus (FPV). Revaccination against FPV, FHV-1, and FCV is r


## 7. Run Both Pipelines on the Test Set

We run each pipeline under a separate LangSmith project to isolate cost tracking.

In [8]:
import time
from copy import deepcopy
from langchain_core.tracers import LangChainTracer

fw_dataset = deepcopy(dataset)
oai_dataset = deepcopy(dataset)


def run_pipeline(rag_graph, eval_dataset, project_name, delay=2, max_retries=3):
    """Run a RAG pipeline over the eval dataset with retry + backoff for rate limits."""
    tracer = LangChainTracer(project_name=project_name)
    print(f"Running {project_name}...")
    for i, row in enumerate(eval_dataset):
        for attempt in range(max_retries):
            try:
                result = rag_graph.invoke(
                    {"question": row.eval_sample.user_input},
                    config={"callbacks": [tracer]},
                )
                row.eval_sample.response = result["response"]
                row.eval_sample.retrieved_contexts = [
                    doc.page_content for doc in result["context"]
                ]
                print(f"  [{i+1}/{len(eval_dataset)}] OK")
                break
            except Exception as e:
                if "429" in str(e) or "RATE_LIMIT" in str(e):
                    wait = delay * (2 ** attempt)
                    print(f"  [{i+1}] Rate limited, retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    raise
        time.sleep(delay)
    print(f"  Done — {len(eval_dataset)} samples")


# Wait for Fireworks rate limit to reset after vector store creation in cell 11
print("Waiting 60s for Fireworks rate limit to reset...")
time.sleep(60)

run_pipeline(fw_rag, fw_dataset, "RAGAS-Fireworks", delay=5, max_retries=5)
run_pipeline(oai_rag, oai_dataset, "RAGAS-OpenAI", delay=1)

Waiting 60s for Fireworks rate limit to reset...
Running RAGAS-Fireworks...
  [1/10] OK
  [2/10] OK
  [3/10] OK
  [4/10] OK
  [5/10] OK
  [6/10] OK
  [7] Rate limited, retrying in 5s...
  [7/10] OK
  [8/10] OK
  [9/10] OK
  [10/10] OK
  Done — 10 samples
Running RAGAS-OpenAI...
  [1/10] OK
  [2/10] OK
  [3/10] OK
  [4/10] OK
  [5/10] OK
  [6/10] OK
  [7/10] OK
  [8/10] OK
  [9/10] OK
  [10/10] OK
  Done — 10 samples


## 8. RAGAS Evaluation

Evaluate both pipelines with the same metrics and judge model.

In [9]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.cost import get_token_usage_for_openai

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", max_tokens=4096))
run_config = RunConfig(timeout=360)

metrics = [LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy()]

/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_48930/2768829667.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_48930/2768829667.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy
/var/folders/g6/fr90m37j7gxbq971zg96s0580000gn/T/ipykernel_48930/2768829667.py:2: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please 

In [10]:
import numpy as np


def clean_for_ragas(testset_dataset) -> "EvaluationDataset":
    """Convert a RAGAS testset to EvaluationDataset, dropping NaN-heavy columns."""
    df = testset_dataset.to_pandas()
    keep = ["user_input", "response", "retrieved_contexts", "reference", "reference_contexts"]
    df = df[[c for c in keep if c in df.columns]]
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: x if x is not None and x is not np.nan and not (isinstance(x, float) and np.isnan(x)) else "")
    return EvaluationDataset.from_pandas(df)


fw_eval_tracer = LangChainTracer(project_name="RAGAS-Eval-Fireworks")

fw_eval_dataset = clean_for_ragas(fw_dataset)
fw_result = evaluate(
    dataset=fw_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_config,
    token_usage_parser=get_token_usage_for_openai,
    callbacks=[fw_eval_tracer],
)
print("Fireworks:", fw_result)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Fireworks: {'context_recall': 0.8933, 'faithfulness': 0.7310, 'factual_correctness(mode=f1)': 0.3270, 'answer_relevancy': 0.9330}


In [11]:
oai_eval_tracer = LangChainTracer(project_name="RAGAS-Eval-OpenAI")

oai_eval_dataset = clean_for_ragas(oai_dataset)
oai_result = evaluate(
    dataset=oai_eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    run_config=run_config,
    token_usage_parser=get_token_usage_for_openai,
    callbacks=[oai_eval_tracer],
)
print("OpenAI:", oai_result)

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[1]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)


OpenAI: {'context_recall': 0.8667, 'faithfulness': 0.9235, 'factual_correctness(mode=f1)': 0.4200, 'answer_relevancy': 0.9607}


## 9. Side-by-Side Comparison

In [12]:
import pandas as pd

fw_avg = fw_result._repr_dict
oai_avg = oai_result._repr_dict

comparison = pd.DataFrame({
    "Metric": list(fw_avg.keys()),
    "Fireworks (gpt-oss-20b)": [round(v, 4) for v in fw_avg.values()],
    "OpenAI (gpt-4.1-mini)": [round(v, 4) for v in oai_avg.values()],
})
comparison["Delta (OAI - FW)"] = comparison["OpenAI (gpt-4.1-mini)"] - comparison["Fireworks (gpt-oss-20b)"]
comparison

,Metric,Fireworks (gpt-oss-20b),OpenAI (gpt-4.1-mini),Delta (OAI - FW)
0,context_recall,0.8933,0.8667,-0.0266
1,faithfulness,0.7310,0.9235,0.1925
2,factual_correctness(mode=f1),0.3270,0.4200,0.0930
3,answer_relevancy,0.9330,0.9607,0.0277


## 10. LangSmith Cost Analysis

LangSmith traces are organized into four projects:

| Project | What it captures |
|---|---|
| `RAGAS-Fireworks` | Fireworks pipeline inference (embed + generate) |
| `RAGAS-OpenAI` | OpenAI pipeline inference (embed + generate) |
| `RAGAS-Eval-Fireworks` | RAGAS judge calls for Fireworks results |
| `RAGAS-Eval-OpenAI` | RAGAS judge calls for OpenAI results |

Open your [LangSmith dashboard](https://smith.langchain.com/) and compare:

- **Token usage** per project (input vs output tokens)
- **Cost per query** across both inference pipelines
- **Total cost** at scale projections

Key things to look for:
- Fireworks uses open-source models which are typically cheaper per token
- OpenAI's `gpt-4.1-mini` may produce higher-quality responses but at a higher cost
- Embedding costs differ: Fireworks `qwen3-embedding-8b` vs OpenAI `text-embedding-3-small`

In [13]:
# gpt-4.1-mini pricing: $0.40/1M input, $1.60/1M output
GPT41_MINI_INPUT = 0.40 / 1_000_000
GPT41_MINI_OUTPUT = 1.60 / 1_000_000

for name, result in [("Fireworks", fw_result), ("OpenAI", oai_result)]:
    tokens = result.total_tokens()
    cost = tokens.input_tokens * GPT41_MINI_INPUT + tokens.output_tokens * GPT41_MINI_OUTPUT
    print(f"{name} eval — input: {tokens.input_tokens}, output: {tokens.output_tokens}, cost: ${cost:.4f}")

fw_detail = fw_result.to_pandas()
fw_detail["provider"] = "Fireworks"

oai_detail = oai_result.to_pandas()
oai_detail["provider"] = "OpenAI"

all_results = pd.concat([fw_detail, oai_detail], ignore_index=True)
# Show available metric columns
metric_cols = [c for c in all_results.columns if c not in ("provider", "user_input", "response", "reference", "retrieved_contexts", "reference_contexts")]
print("Available metric columns:", metric_cols)
all_results[["provider", "user_input"] + metric_cols]

Fireworks eval — input: 108098, output: 50973, cost: $0.1248
OpenAI eval — input: 109605, output: 46205, cost: $0.1178
Available metric columns: ['context_recall', 'faithfulness', 'factual_correctness(mode=f1)', 'answer_relevancy']


,provider,user_input,context_recall,faithfulness,factual_correctness(mode=f1),answer_relevancy
0,Fireworks,"In Texas cats, what kind of health stuff do we...",1.000000,0.578947,0.05,0.875356
1,Fireworks,What is the significance of the AAFP guideline...,0.600000,0.739130,0.67,0.947933
2,Fireworks,What is the significance of JAAHA in the conte...,1.000000,0.285714,0.20,0.848202
3,Fireworks,What is AAHA?,1.000000,1.000000,0.00,0.958064
4,Fireworks,What does AAHA stand for?,1.000000,1.000000,0.00,1.000000
5,Fireworks,What the American Association of Feline Practi...,1.000000,0.842105,0.42,0.914833
6,Fireworks,How do environmental changes help cats with ur...,1.000000,0.703704,0.64,0.956718
7,Fireworks,How can I tell if my cat is eating and drinkin...,1.000000,0.695652,0.39,0.935978
8,Fireworks,How does DJD relate to environmental managemen...,0.666667,0.529412,0.40,0.901723
9,Fireworks,How can environmental modifications and gentle...,0.666667,0.935484,0.50,0.991423
